In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import random
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ → robomaster-cv/
DATA_DIR = PROJECT_ROOT / 'data/mergeRM'
DATASET_YAML = str(PROJECT_ROOT / 'configs/data.yaml')

IMG_DIR = "/home/jupyter-robomaster/jithendra/data/mergeRM/images/test"
LBL_DIR = "/home/jupyter-robomaster/jithendra/data/mergeRM/labels/test"

def draw_obb(img, label_path):
    h, w = img.shape[:2]
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls_id = int(parts[0])
            coords = np.array([float(x) for x in parts[1:9]])
            coords = coords.reshape(4, 2)
            coords[:, 0] *= w
            coords[:, 1] *= h
            pts = coords.astype(np.int32)
            cv2.polylines(img, [pts], True, (0, 255, 0), 2)
            cv2.putText(img, f"plate", (pts[0][0], pts[0][1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return img

# Pick 6 random images that have labels
images = [f for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png'))]
random.shuffle(images)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
count = 0
for img_name in images:
    if count >= 6:
        break
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    lbl_path = os.path.join(LBL_DIR, lbl_name)
    if not os.path.exists(lbl_path) or os.path.getsize(lbl_path) == 0:
        continue
    
    img = cv2.imread(os.path.join(IMG_DIR, img_name))
    img = draw_obb(img, lbl_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    ax = axes[count // 3][count % 3]
    ax.imshow(img)
    ax.set_title(img_name, fontsize=10)
    ax.axis('off')
    count += 1

plt.suptitle("Ground Truth OBB Labels", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO

best_model = YOLO("/home/jupyter-robomaster/jithendra/notebooks/runs/obb/obb_final_model/weights/best.pt")

# Pick same images or new ones
sample_images = random.sample([f for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png'))], 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, img_name in enumerate(sample_images):
    img_path = os.path.join(IMG_DIR, img_name)
    results = best_model.predict(img_path, conf=0.25, verbose=False)
    
    plotted = results[0].plot()  # draws predictions on image
    plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
    
    ax = axes[i // 3][i % 3]
    ax.imshow(plotted)
    ax.set_title(f"{img_name} ({len(results[0].obb)} detections)", fontsize=10)
    ax.axis('off')

plt.suptitle("Model Predictions (conf > 0.25)", fontsize=14)
plt.tight_layout()
plt.show()

In [4]:
import cv2
import numpy as np
import os
from pathlib import Path
from ultralytics import YOLO
from shapely.geometry import Polygon

# Setup
best_model = YOLO("/home/jupyter-robomaster/jithendra/notebooks/runs/obb/obb_final_model/weights/best.pt")
IMG_DIR = "/home/jupyter-robomaster/jithendra/data/mergeRM/images/test"
LBL_DIR = "/home/jupyter-robomaster/jithendra/data/mergeRM/labels/test"
OUT_DIR = "/home/jupyter-robomaster/jithendra/error_analysis"
CONF_THRESH = 0.25
IOU_THRESH = 0.5

# Create output folders
for folder in ['false_positives', 'false_negatives', 'true_positives']:
    os.makedirs(os.path.join(OUT_DIR, folder), exist_ok=True)


In [5]:

def get_gt_boxes(label_path, w, h):
    """Load ground truth OBB boxes from label file."""
    boxes = []
    if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 9:
                    coords = np.array([float(x) for x in parts[1:9]]).reshape(4, 2)
                    coords[:, 0] *= w
                    coords[:, 1] *= h
                    boxes.append(coords)
    return boxes

def obb_iou(box1, box2):
    """Compute IoU between two OBB boxes using Shapely polygons."""
    try:
        p1 = Polygon(box1)
        p2 = Polygon(box2)
        if not p1.is_valid or not p2.is_valid:
            return 0.0
        inter = p1.intersection(p2).area
        union = p1.union(p2).area
        return inter / union if union > 0 else 0.0
    except:
        return 0.0

def draw_boxes(img, boxes, color, label):
    """Draw OBB boxes on image."""
    for box in boxes:
        pts = box.astype(np.int32)
        cv2.polylines(img, [pts], True, color, 2)
        cv2.putText(img, label, (pts[0][0], pts[0][1] - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img

# Run analysis
fp_count, fn_count, tp_count = 0, 0, 0
fp_images, fn_images = [], []

images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png'))])

for img_name in images:
    img_path = os.path.join(IMG_DIR, img_name)
    lbl_path = os.path.join(LBL_DIR, os.path.splitext(img_name)[0] + '.txt')

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Get ground truth
    gt_boxes = get_gt_boxes(lbl_path, w, h)

    # Get predictions
    results = best_model.predict(img_path, conf=CONF_THRESH, verbose=False)
    pred_boxes = []
    pred_confs = []
    if results[0].obb is not None and len(results[0].obb) > 0:
        for i in range(len(results[0].obb)):
            xyxyxyxy = results[0].obb.xyxyxyxy[i].cpu().numpy()
            conf = results[0].obb.conf[i].cpu().numpy()
            pred_boxes.append(xyxyxyxy)
            pred_confs.append(float(conf))

    # Match predictions to ground truth
    gt_matched = [False] * len(gt_boxes)
    pred_matched = [False] * len(pred_boxes)

    for pi, pbox in enumerate(pred_boxes):
        best_iou = 0
        best_gi = -1
        for gi, gbox in enumerate(gt_boxes):
            if gt_matched[gi]:
                continue
            iou = obb_iou(pbox, gbox)
            if iou > best_iou:
                best_iou = iou
                best_gi = gi
        if best_iou >= IOU_THRESH and best_gi >= 0:
            gt_matched[best_gi] = True
            pred_matched[pi] = True
            tp_count += 1

    # False positives: unmatched predictions
    img_fps = [pred_boxes[i] for i in range(len(pred_boxes)) if not pred_matched[i]]
    fp_confs = [pred_confs[i] for i in range(len(pred_confs)) if not pred_matched[i]]

    # False negatives: unmatched ground truth
    img_fns = [gt_boxes[i] for i in range(len(gt_boxes)) if not gt_matched[i]]

    fp_count += len(img_fps)
    fn_count += len(img_fns)

    # Save false positive images
    if len(img_fps) > 0:
        vis = img.copy()
        vis = draw_boxes(vis, gt_boxes, (0, 255, 0), "GT")
        for i, fp_box in enumerate(img_fps):
            pts = fp_box.astype(np.int32)
            cv2.polylines(vis, [pts], True, (0, 0, 255), 2)
            cv2.putText(vis, f"FP:{fp_confs[i]:.2f}", (pts[0][0], pts[0][1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        cv2.imwrite(os.path.join(OUT_DIR, 'false_positives', img_name), vis)
        fp_images.append(img_name)

    # Save false negative images
    if len(img_fns) > 0:
        vis = img.copy()
        vis = draw_boxes(vis, [gt_boxes[i] for i in range(len(gt_boxes)) if gt_matched[i]], (0, 255, 0), "TP")
        vis = draw_boxes(vis, img_fns, (0, 165, 255), "MISSED")
        # Also draw any predictions
        for i, pbox in enumerate(pred_boxes):
            pts = pbox.astype(np.int32)
            color = (255, 0, 0) if pred_matched[i] else (0, 0, 255)
            label = f"P:{pred_confs[i]:.2f}" if pred_matched[i] else f"FP:{pred_confs[i]:.2f}"
            cv2.polylines(vis, [pts], True, color, 2)
            cv2.putText(vis, label, (pts[0][0], pts[0][1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        cv2.imwrite(os.path.join(OUT_DIR, 'false_negatives', img_name), vis)
        fn_images.append(img_name)

print(f"\n{'='*40}")
print(f"ERROR ANALYSIS SUMMARY")
print(f"{'='*40}")
print(f"True Positives:  {tp_count}")
print(f"False Positives: {fp_count} ({len(fp_images)} images)")
print(f"False Negatives: {fn_count} ({len(fn_images)} images)")
print(f"Precision:       {tp_count/(tp_count+fp_count):.4f}")
print(f"Recall:          {tp_count/(tp_count+fn_count):.4f}")
print(f"\nImages saved to: {OUT_DIR}")
print(f"  false_positives/ - Red boxes = FP, Green = GT")
print(f"  false_negatives/ - Orange 'MISSED' = FN, Blue = matched pred")


ERROR ANALYSIS SUMMARY
True Positives:  828
False Positives: 242 (156 images)
False Negatives: 99 (68 images)
Precision:       0.7738
Recall:          0.8932

Images saved to: /home/jupyter-robomaster/jithendra/error_analysis
  false_positives/ - Red boxes = FP, Green = GT
  false_negatives/ - Orange 'MISSED' = FN, Blue = matched pred
